## Hashing

[Video](https://youtu.be/Ll6Bp4eoT2A) - [Aufgabenvideo](https://youtu.be/oMT9RXqmB4w)

Wir wollen Objekte abspeichern und schnell wiederfinden. In einer Liste müssen wir dafür im schlimmsten Fall jedes
Element ansehen, das dauert $O(n)$. In einer sortierten Liste mit binärer Suche oder in einem ausgeglichenen Suchbaum
geht es in $O(\log n)$. Hashing geht einen anderen Weg: Aus dem Objekt selbst wird **berechnet**, an welcher Stelle es
liegt. Im günstigen Fall genügt dann ein einziger Zugriff.

#### Die Hashfunktion

Zum Abspeichern und Wiederfinden von Objekten wäre folgende Funktion hilfreich:

$f$: Objekte $\rightarrow \{0, 1, \dots, N-1\}$

Dann könnte Objekt $x$ unter Adresse $f(x)$ gespeichert werden. $f$ heißt **Hashfunktion**, $f(x)$ der **Hashwert**
von $x$ und die Liste mit den $N$ Speicherplätzen die **Hashtabelle**.

Problem: Die Anzahl der möglichen Objekte ist meist viel größer als die Anzahl der Adressen. Dann müssen sich
mehrere Objekte eine Adresse teilen.

<img src='bild1.png'>

**Kollision:** $f(x) = f(y)$ für $x \ne y$.

Kollisionen lassen sich also nicht ganz vermeiden. Eine gute Hashfunktion

- ist schnell zu berechnen,
- liefert für dasselbe Objekt immer denselben Wert,
- verteilt die Objekte möglichst gleichmäßig auf die Adressen, damit es wenige Kollisionen gibt.

#### Ein erstes Beispiel mit Zahlen

Wir haben $N = 10$ Plätze und nehmen als Hashfunktion $f(x) = x \% 10$, also den Rest bei der Division durch 10.
Der Hashwert ist die letzte Ziffer der Zahl.

In [1]:
def f(x):
    return x % 10

for x in [12, 18, 35, 22]:
    print(x, '->', f(x))

12 -> 2
18 -> 8
35 -> 5
22 -> 2


Die 12 und die 22 wollen beide auf Platz 2. Das ist eine Kollision. Wie man damit umgeht, sehen wir weiter unten.

#### Eine Hashfunktion für Strings

Gegeben seien die Adressen von $0$ bis $N-1$.

$f$: Menge der Strings $\rightarrow \{0, \dots, N-1\}$

$f(s)$ = (Summe aller Unicode-Werte der Zeichen von $s$) % $N$

Für beliebige Objekte $x$ könnte $f$ einen Wert für die String-Darstellung von $x$ berechnen. Diese Hashfunktion
funktioniert in vielen Fällen schon ganz gut, wenn $N$ eine Primzahl ist, die nicht nahe bei Potenzen von 2 oder 10
liegt. Dadurch erreicht man eine einigermaßen gleichmäßige, zufallsartige Verteilung. Wir nehmen $N = 7643$:

- 7643 ist eine Primzahl,
- die nächsten Zweierpotenzen sind 4096 und 8192,
- die nächsten Zehnerpotenzen sind 1000 und 10000.

In [2]:
def simple_hash(s):
    hash_value = 0
    for c in s:
        hash_value = (hash_value + ord(c)) % 7643
    return hash_value

In [3]:
a = ['hello', 'world', 'apple', 'banana', 'orange', 'grape', 'melon']
for s in a:
    print(s, simple_hash(s))

hello 532
world 552
apple 530
banana 609
orange 636
grape 527
melon 539


In [4]:
b = ['abcd', 'bcda', 'acbd', 'abdc', 'abcd1', 'abcd2']
for s in b:
    print(s, simple_hash(s))

abcd 394
bcda 394
acbd 394
abdc 394
abcd1 443
abcd2 444


Strings mit den gleichen Zeichen ergeben den gleichen Hashwert, denn bei der Addition spielt die Reihenfolge keine
Rolle. Das vermeiden wir, indem wir den bisherigen Wert vor jedem Zeichen mit einer Zahl, hier 17, multiplizieren.

Das ist dieselbe Rechnung wie beim Umrechnen einer Zahl aus einem Stellenwertsystem: Die Ziffernfolge 3, 0, 7 ergibt
im Zehnersystem $(3 \cdot 10 + 0) \cdot 10 + 7 = 307$, die Folge 7, 0, 3 dagegen 703. Hier sind die Unicode-Werte die
Ziffern und 17 die Basis. Damit zählt auch die Stelle, an der ein Zeichen steht.

In [5]:
def simple_hash(s):
    hash_value = 0
    for c in s:
        hash_value = (hash_value * 17 + ord(c)) % 7643
    return hash_value

In [6]:
for s in a + b:
    print(s, simple_hash(s))

hello 5741
world 2456
apple 3650
banana 2333
orange 1805
grape 5897
melon 3031
abcd 2228
bcda 7444
acbd 2500
abdc 2244
abcd1 7353
abcd2 7354


#### Umgang mit Kollisionen

Für den Umgang mit Kollisionen gibt es zwei Techniken:

- **Offenes Hashing:** Auf jedem Platz der Tabelle steht eine Liste. Alle Objekte mit demselben Hashwert kommen in
  diese Liste.
- **Geschlossenes Hashing:** Auf jedem Platz steht höchstens ein Objekt. Ist der Platz $f(x)$ schon belegt, sucht man
  nach festen Regeln einen anderen freien Platz in der Tabelle.

Vorsicht bei englischen Texten: Dort heißt das geschlossene Hashing *open addressing*, das offene Hashing
*separate chaining*.

#### Offenes Hashing

<img src='hashing01.png' width='301'>

Datenstruktur für die Implementierung: eine Liste (Array) von verketteten Listen.

- **Einfügen:** $x$ wird an die Liste auf Platz $f(x)$ angehängt.
- **Suchen:** Man geht nur die Liste auf Platz $f(x)$ durch.

Sind die Objekte gleichmäßig verteilt und gibt es nicht viel mehr Objekte als Plätze, dann sind die Listen kurz und
Einfügen und Suchen gehen schnell.

In Python nehmen wir für die verketteten Listen einfach Python-Listen. Achtung: `[[]] * 10` würde zehnmal **dieselbe**
Liste enthalten. Wir brauchen zehn verschiedene Listen.

In [7]:
tabelle = [[] for i in range(10)]

def einfuegen_offen(tabelle, x):
    liste = tabelle[f(x)]
    if x not in liste:
        liste.append(x)

def enthalten_offen(tabelle, x):
    return x in tabelle[f(x)]

for x in [12, 18, 22, 35, 42, 19]:
    einfuegen_offen(tabelle, x)

for i in range(10):
    print(i, tabelle[i])

0 []
1 []
2 [12, 22, 42]
3 []
4 []
5 [35]
6 []
7 []
8 [18]
9 [19]


In [8]:
print(enthalten_offen(tabelle, 42), enthalten_offen(tabelle, 32))

True False


Bei der Suche nach der 32 muss man nur die drei Zahlen auf Platz 2 ansehen, nicht alle sechs.

#### Geschlossenes Hashing

<img src='hashing02.png' width='400'>

Datenstruktur für die Implementierung: eine Liste für die Objekte und eine für die Zustände (leer, belegt,
gelöscht). Wozu man den Zustand *gelöscht* braucht, sehen wir weiter unten.

Ist der Platz $y = f(x)$ schon belegt, sucht man für $x$ einen Alternativplatz. Das nennt man **Sondieren**:

- **lineares Sondieren:** $y+1, y+2, y+3, y+4, \dots$
- **quadratisches Sondieren:** $y+1, y+4, y+9, y+16, \dots$
- **Double Hashing:** $y + f_2(x), y + 2 \cdot f_2(x), y + 3 \cdot f_2(x), \dots$ Die Schrittweite wird durch eine
  zweite Hashfunktion $f_2$ bestimmt.

Alle Berechnungen werden jeweils modulo $N$ durchgeführt. Am Ende der Tabelle geht es also vorne weiter.

Eine **perfekte** Hashfunktion verursacht keine Kollisionen.

In Python schreiben wir das Sondieren so, dass das Verfahren als Funktion übergeben wird. Die Funktion bekommt das
Objekt $x$ und die Nummer $i$ des Versuchs und gibt an, wie weit man von $f(x)$ aus weitergeht. Freie Plätze
enthalten `None`.

In [9]:
def linear(x, i):
    return i

def quadratisch(x, i):
    return i * i

def einfuegen(tabelle, x, sondieren):
    '''fügt x ein und gibt den Platz zurück, None wenn kein Platz gefunden wurde'''
    N = len(tabelle)
    for i in range(N):
        p = (f(x) + sondieren(x, i)) % N
        if tabelle[p] is None:
            tabelle[p] = x
            return p
    return None

def zeige(tabelle):
    for i in range(len(tabelle)):
        print(f'{i:>4}', end='')
    print()
    for x in tabelle:
        print(f'{"." if x is None else x:>4}', end='')
    print()

#### Lineares Sondieren

In [10]:
folge = [31, 44, 51, 24, 61, 9, 39]
tabelle = [None] * 10
for x in folge:
    p = einfuegen(tabelle, x, linear)
    print(f'{x:>3}  f(x) = {f(x)}  ->  Platz {p}')
print()
zeige(tabelle)

 31  f(x) = 1  ->  Platz 1
 44  f(x) = 4  ->  Platz 4
 51  f(x) = 1  ->  Platz 2
 24  f(x) = 4  ->  Platz 5
 61  f(x) = 1  ->  Platz 3
  9  f(x) = 9  ->  Platz 9
 39  f(x) = 9  ->  Platz 0

   0   1   2   3   4   5   6   7   8   9
  39  31  51  61  44  24   .   .   .   9


Die 61 findet die Plätze 1 und 2 belegt und landet auf Platz 3. Die 39 findet Platz 9 belegt, geht über das Ende der
Tabelle hinaus und landet auf Platz 0.

**Suchen** geht genauso wie Einfügen: Man beginnt bei $f(x)$ und sondiert, bis man $x$ gefunden hat. Trifft man
vorher auf einen **leeren** Platz, kann $x$ nicht in der Tabelle sein, denn sonst wäre es spätestens dort eingefügt
worden.

In [11]:
def suchen(tabelle, x, sondieren):
    '''gibt den Platz von x zurück, None wenn x nicht in der Tabelle ist'''
    N = len(tabelle)
    for i in range(N):
        p = (f(x) + sondieren(x, i)) % N
        print(p, end=' ')
        if tabelle[p] is None:
            return None
        if tabelle[p] == x:
            return p
    return None

print('->', suchen(tabelle, 39, linear))
print('->', suchen(tabelle, 29, linear))

9 0 -> 0
9 0 1 2 3 4 5 6 -> None


Für die 39 werden zwei Plätze angesehen. Für die 29, die nicht in der Tabelle steht, sind es acht Plätze, bis man
auf den leeren Platz 6 trifft.

Das zeigt den Nachteil des linearen Sondierens: Es bilden sich lange Ketten belegter Plätze, man spricht von
**Häufung** (englisch *cluster*). Jede Zahl, deren Hashwert in die Kette fällt, wird an ihr Ende gesetzt und macht die
Kette noch länger. Die Plätze 9, 0, 1, 2, 3, 4, 5 bilden hier schon eine solche Kette.

#### Quadratisches Sondieren

Beim quadratischen Sondieren springt man mit wachsenden Abständen weiter: $y+1, y+4, y+9, \dots$ Dadurch wachsen die
Ketten nicht so leicht zusammen.

In [12]:
tabelle = [None] * 10
for x in folge:
    p = einfuegen(tabelle, x, quadratisch)
    print(f'{x:>3}  f(x) = {f(x)}  ->  Platz {p}')
print()
zeige(tabelle)

 31  f(x) = 1  ->  Platz 1
 44  f(x) = 4  ->  Platz 4
 51  f(x) = 1  ->  Platz 2
 24  f(x) = 4  ->  Platz 5
 61  f(x) = 1  ->  Platz 0
  9  f(x) = 9  ->  Platz 9
 39  f(x) = 9  ->  Platz 3

   0   1   2   3   4   5   6   7   8   9
  61  31  51  39  44  24   .   .   .   9


Die 61 findet die Plätze 1, 2 und $1+4=5$ belegt und kommt auf $1+9=10$, also Platz 0.

Beim quadratischen Sondieren werden aber nicht immer alle Plätze besucht. Bei $N = 10$ liefern die Quadratzahlen
modulo 10 nur sechs verschiedene Abstände:

In [13]:
print(sorted({i * i % 10 for i in range(10)}))

[0, 1, 4, 5, 6, 9]


Die Abstände 2, 3, 7 und 8 kommen nie vor. Sind die anderen Plätze belegt, findet eine Zahl keinen Platz, obwohl noch
vier Plätze frei sind:

In [14]:
tabelle = [None] * 10
for x in [10, 11, 14, 15, 16, 19]:
    einfuegen(tabelle, x, quadratisch)
zeige(tabelle)
print(einfuegen(tabelle, 30, quadratisch))

   0   1   2   3   4   5   6   7   8   9
  10  11   .   .  14  15  16   .   .  19
None


Ist $N$ eine Primzahl, dann erreicht man beim quadratischen Sondieren mindestens die Hälfte der Plätze. Solange die
Tabelle höchstens halb voll ist, findet man also immer einen freien Platz.

#### Double Hashing

Beim Double Hashing bestimmt eine zweite Hashfunktion $f_2$ die Schrittweite. Zahlen mit demselben Hashwert $f(x)$
haben meist verschiedene Schrittweiten und gehen deshalb verschiedene Wege durch die Tabelle. Wir nehmen
$f_2(x) = (2x + 3) \% 10$.

In [15]:
def f2(x):
    return (2 * x + 3) % 10

def double(x, i):
    return i * f2(x)

tabelle = [None] * 10
for x in folge:
    p = einfuegen(tabelle, x, double)
    print(f'{x:>3}  f(x) = {f(x)}  f2(x) = {f2(x)}  ->  Platz {p}')
print()
zeige(tabelle)

 31  f(x) = 1  f2(x) = 5  ->  Platz 1
 44  f(x) = 4  f2(x) = 1  ->  Platz 4
 51  f(x) = 1  f2(x) = 5  ->  Platz 6
 24  f(x) = 4  f2(x) = 1  ->  Platz 5
 61  f(x) = 1  f2(x) = 5  ->  Platz None
  9  f(x) = 9  f2(x) = 1  ->  Platz 9
 39  f(x) = 9  f2(x) = 1  ->  Platz 0

   0   1   2   3   4   5   6   7   8   9
  39  31   .   .  44  24  51   .   .   9


Die 61 findet keinen Platz, obwohl die Tabelle nur halb voll ist. Ihre Schrittweite ist 5. Von Platz 1 aus erreicht
sie nur die Plätze 1 und 6, und beide sind belegt.

Die Schrittweite darf also nicht 0 sein und muss zu $N$ passen: Haben Schrittweite und $N$ einen gemeinsamen Teiler,
wird nur ein Teil der Plätze besucht. Am einfachsten wählt man für $N$ eine Primzahl, dann erreicht jede Schrittweite
von 1 bis $N-1$ alle Plätze.

In [16]:
print([(1 + i * 5) % 10 for i in range(10)])
print([(1 + i * 5) % 11 for i in range(11)])

[1, 6, 1, 6, 1, 6, 1, 6, 1, 6]
[1, 6, 0, 5, 10, 4, 9, 3, 8, 2, 7]


#### Löschen beim geschlossenen Hashing

Beim Löschen darf man den Platz nicht einfach leeren. Wir sehen uns das in der Tabelle mit linearem Sondieren an und
löschen die 51 auf Platz 2.

In [17]:
tabelle = [None] * 10
for x in folge:
    einfuegen(tabelle, x, linear)
tabelle[2] = None
zeige(tabelle)
print('->', suchen(tabelle, 61, linear))

   0   1   2   3   4   5   6   7   8   9
  39  31   .  61  44  24   .   .   .   9
1 2 -> None


Die Suche nach der 61 beginnt bei Platz 1, trifft dann auf den leeren Platz 2 und hört auf. Die 61 auf Platz 3 wird
nicht mehr gefunden.

Deshalb merkt man sich für jeden Platz einen Zustand: **leer**, **belegt** oder **gelöscht** (L, B, G im Bild oben).
Ein gelöschter Platz beendet die Suche nicht, man sondiert weiter. Beim Einfügen darf ein gelöschter Platz aber wieder
belegt werden. Statt einer eigenen Liste für die Zustände schreiben wir hier auf gelöschte Plätze die Markierung
`'G'`.

In [18]:
tabelle = [None] * 10
for x in folge:
    einfuegen(tabelle, x, linear)

def loeschen(tabelle, x, sondieren):
    p = suchen(tabelle, x, sondieren)
    if p is not None:
        tabelle[p] = 'G'

loeschen(tabelle, 51, linear)
print()
zeige(tabelle)
print('->', suchen(tabelle, 61, linear))

1 2 
   0   1   2   3   4   5   6   7   8   9
  39  31   G  61  44  24   .   .   .   9
1 2 3 -> 3


Damit beim Einfügen gelöschte Plätze wiederverwendet werden, ändert man die Bedingung in `einfuegen` zu
`tabelle[p] is None or tabelle[p] == 'G'`. Vorher muss man dann aber prüfen, ob $x$ schon in der Tabelle steht, sonst
könnte es doppelt hineinkommen.

#### Laufzeit bei geschlossenem Hashing

$N$ = Anzahl der möglichen Speicherpositionen, $n$ = Anzahl der gespeicherten Objekte

$\alpha = \frac{n}{N}$ heißt **Belegungsgrad** oder Auslastungsfaktor.

Die Laufzeit hängt nicht von $n$ ab, sondern nur vom Belegungsgrad. Als durchschnittliche Anzahl der
Sondierungsschritte bei Double Hashing ergibt sich bei

- erfolgloser Suche: $\approx \frac{1}{1-\alpha}$, also $5$ für $\alpha = 0.8$,
- erfolgreicher Suche: $\approx -\frac{\ln(1-\alpha)}{\alpha}$, also $2.01$ für $\alpha = 0.8$.

D.h. in 2 Schritten wird von 800.000 Elementen aus einer 1.000.000 großen Tabelle das richtige gefunden. Ein
ausgeglichener Suchbaum benötigt dafür etwa 20 Vergleiche.

<img src='bild4.png' width='500'>

In [19]:
import math

for alpha in [0.5, 0.8, 0.9, 0.95, 0.99]:
    erfolglos = 1 / (1 - alpha)
    erfolgreich = -math.log(1 - alpha) / alpha
    print(f'{alpha:<5} {erfolglos:6.1f} {erfolgreich:6.2f}')

0.5      2.0   1.39
0.8      5.0   2.01
0.9     10.0   2.56
0.95    20.0   3.15
0.99   100.0   4.65


Wird die Tabelle fast voll, steigt die Zahl der Schritte stark an. Man legt dann eine größere Tabelle an, zum Beispiel
ungefähr doppelt so groß, und fügt alle Objekte neu ein. Deshalb wächst der Speicherbedarf in Sprüngen.

Die Laufzeit $O(1)$ gilt im Mittel. Im schlechtesten Fall haben alle Objekte denselben Hashwert, dann dauert eine
Suche $O(n)$ wie in einer Liste. Bei einer guten Hashfunktion kommt das praktisch nicht vor.

|                     | ausgeglichener Suchbaum | geschlossenes Hashing |
|---------------------|-------------------------|-----------------------|
| Laufzeit im Mittel  | $O(\log n)$             | $O(1)$                |
| schlechtester Fall  | $O(\log n)$             | $O(n)$                |
| Speicherbedarf      | dynamisch wachsend      | in Sprüngen wachsend  |
| Sortierung          | durch Traversierung     | -                     |

#### Hashing in Python

Die Python-Datentypen `set` und `dict` sind Hashtabellen. Deshalb ist `x in s` bei einem Set so schnell, und deshalb
liefern Sets ihre Elemente nicht sortiert. Die eingebaute Funktion `hash` berechnet den Hashwert eines Objekts.

In [20]:
print(hash(42))
print(hash('Susi'))
print(hash((1, 2)))

42
-4160361539719830439
-3550055125485641917


Ganze Zahlen sind ihr eigener Hashwert. Der Hashwert eines Strings ändert sich bei jedem Start von Python, innerhalb
eines Programmlaufs bleibt er gleich.

Listen haben keinen Hashwert:

In [21]:
hash([1, 2])

TypeError: unhashable type: 'list'

Eine Liste ist veränderbar. Würde sie in einem Set gespeichert und danach verändert, dann hätte sie einen anderen
Hashwert, stünde also am falschen Platz und würde nicht mehr gefunden. Deshalb dürfen in Sets und als Schlüssel in
Dictionaries nur unveränderbare Objekte wie Zahlen, Strings und Tupel stehen.

#### Hashfunktionen in der Datensicherheit

In der Datensicherheit werden **kryptographische Hashfunktionen** wie SHA-256 verwendet, zum Beispiel für
Signaturen und Passwörter. Sie haben strengere Anforderungen: Aus dem Hashwert darf man nicht auf das Objekt
zurückschließen können, und es muss praktisch unmöglich sein, zwei Objekte mit demselben Hashwert zu finden. Für
eine Hashtabelle wären sie zu langsam. Mehr dazu in der Einheit Datensicherheit 2.